In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import scanpy as scp
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from mpl_toolkits.axes_grid.inset_locator import inset_axes
import matplotlib.patches as mpatches
import igraph
import scipy.stats as st
import bbknn

from sklearn.decomposition import TruncatedSVD
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neighbors import KNeighborsClassifier

# Whole

In [5]:
adata_embryo = scp.read("results/embryo/Embryo_corrected.h5ad")
adata_gastruloid = scp.read("results/QC.h5ad")

In [6]:
common_genes_embryo = [i in adata_gastruloid.var.index.values for i in adata_embryo.var.index.values]
common_genes_gastruloids = [i in adata_embryo.var.index.values for i in adata_gastruloid.var.index.values]

In [7]:
#Get to the common number of genes
adata_embryo = adata_embryo[:,common_genes_embryo]
adata_gastruloid = adata_gastruloid[:,common_genes_gastruloids]

## PCA embryo

In [8]:
scp.pp.highly_variable_genes(adata_embryo,flavor="seurat")
X = adata_embryo.X[:,adata_embryo.var.loc[:,"highly_variable"]]

Trying to set attribute `.uns` of view, copying.


In [9]:
m = TruncatedSVD(12)
m.fit(X)
X_pca_embryo = m.transform(X)

## Project gastruloids over PCA of the embryo

In [10]:
X = adata_gastruloid.X[:,adata_embryo.var.loc[:,"highly_variable"]]

In [11]:
X_pca_gastruloid = m.transform(X)

## Project over the UMAP

In [12]:
UMAP_embryo = adata_embryo.obsm["X_umap_original"]

m = KNeighborsRegressor(3,metric="correlation")
m.fit(X_pca_embryo,UMAP_embryo)

KeyError: 'X_umap_original'

In [ ]:
UMAP_gastruloid = m.predict(X_pca_gastruloid)

In [ ]:
fig,ax = plt.subplots(figsize=[10,10])

hue = adata_embryo.obs.loc[:,"Louvain Cluster"].astype(int).astype(str)
sns.scatterplot(x=UMAP_embryo[:,0],y=UMAP_embryo[:,1],hue=hue,ax=ax)
sns.scatterplot(x=UMAP_gastruloid[:,0],y=UMAP_gastruloid[:,1],color="black",ax=ax)

fig.savefig("plots/global/embryo_umap_projection.png",bbox_inches="tight")

## Classify them in clusters

In [ ]:
cluster_embryo = adata_embryo.obs.loc[:,["Louvain Cluster"]].astype(int).astype(str)

m = KNeighborsClassifier(3,metric="correlation")
m.fit(X_pca_embryo,cluster_embryo)

In [ ]:
cluster_gastruloid = m.predict(X_pca_gastruloid)

In [ ]:
adata_gastruloid.obs["clusters_from_embryo"] = cluster_gastruloid

In [ ]:
matrix = adata_gastruloid.obs.groupby(["leiden_global","clusters_from_embryo"]).count().iloc[:,0].unstack().transpose()
matrix = matrix/matrix.sum(axis=0)

In [ ]:
fig,ax = plt.subplots(figsize=[10,10])
sns.heatmap(matrix,annot=True,ax=ax)

fig.savefig("plots/sorted/embryo_clusters_overlap.png",bbox_inches="tight")